In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = Path("/home/ubuntu/buspulse/tables/gold-layer")          # folder that holds the .parquet files
OUT_DIR = Path("/home/ubuntu/buspulse/tables/result")       # cleaned outputs are written here
OUT_DIR.mkdir(exist_ok=True)
LOCAL_TZ = "Europe/Oslo"      # matches agency_timezone in gtfs_agency

# Which service dates of the real-time feed to analyse. None = every date found in the feed.
ET_DATES = None               # e.g. ["2026-10-24"]``

def pq_path(name):
    return DATA_DIR / f"{name}.parquet"
import gc
import pyarrow.dataset as ds


In [6]:

rows = []
for p in sorted(DATA_DIR.glob("*.parquet")):        # rglob("*.parquet") to include subfolders
    pf = pq.ParquetFile(p)
    rows.append({
        "file": p.stem,
        "rows": pf.metadata.num_rows,
        "cols": pf.metadata.num_columns,
        "row_groups": pf.metadata.num_row_groups,
        "columns": ", ".join(pf.schema_arrow.names),
    })
overview = pd.DataFrame(rows)
display(overview[["file", "rows", "cols", "row_groups"]])

# Full column lists
for r in rows:
    print(f"{r['file']:<20} {r['columns']}")

,file,rows,cols,row_groups
0,dim_date,23,6,1
1,dim_hour_band,24,2,1
2,dim_line,275,3,1
3,dim_stop,4552,4,1
4,fact_stop_visits,2864058,26,24
5,gold_buckets,14,5,1
6,gold_bunching_stops,2946,20,1
7,gold_busy_roads,5649,30,1
8,gold_data_quality,52,4,1
9,gold_delay_hour,1317540,26,11


dim_date             service_date, weekday, weekday_name, is_weekend, iso_week, iso_year
dim_hour_band        local_hour, hour_band
dim_line             line_ref, route_short_name, route_type_name
dim_stop             stop_ref, stop_name, stop_lat, stop_lon
fact_stop_visits     service_date, local_hour, local_quarter, local_time_bin, hour_band, weekday, journey_ref, line_ref, route_short_name, direction_ref, stop_order, stop_ref, stop_name, stop_lat, stop_lon, aimed_arrival, actual_arrival, aimed_departure, actual_departure, visit_time, delay_s, delay_arrival_s, delay_departure_s, is_actual, delay_outlier, valid
gold_buckets         metric, bucket, lo, hi, label
gold_bunching_stops  stop_ref, n_lines, n_headways, sum_headway_s, sum_sched_headway_s, sum_headway_sq, sum_sched_headway_sq, sum_abs_deviation_s, n_bunched, n_large_gap, avg_headway_s, avg_sched_headway_s, headway_cv, excess_wait_s, pct_bunched, pct_large_gap, stop_name, stop_lat, stop_lon, bunching_rank
gold_busy_roads      f